# Notebook for causal analysis for the INCOME subgroups using DoWhy-library


In [1]:
from dowhy import CausalModel
import pandas as pd

### Define outcome and the confounders for each feature

In [2]:
outcome = "How happy are you?"

In [3]:
feature_confounder_map = {
    "Health condition": [
        "Age",
        "Income quartiles",
        "Chronic health problems?",
        "Education completed",
        "Employment - 7 groups"
    ],

    "I generally feel that what I do in life is worthwhile": [
        "A person to get support from when feeling depressed",
        "How frequently participate in social activities?",
        "Employment - 7 groups",
        "Marital status",
        "Health condition"
    ],

    "Can't find the way because life has become so complicated?": [
        "Education completed",
        "Employment - 7 groups",
        "A person to get support from when feeling depressed",
        "Age",
        "Household size"
    ],

    "I feel I am free to decide how to live my life": [
        "Income quartiles",
        "Education completed",
        "How much trust the government?"
    ],

    "I am optimistic about the future": [
        "Personal financial situation",
        "Access to recreational or green areas?",
        "A person to get support from to raise emergency money",
        "Health condition",
        "Age",
        "Employment - 7 groups"
    ],

    "Household able to make ends meet?": [
        "Employment - 7 groups",
        "Income quartiles",
        "Personal financial situation",
        "Household size",
        "No. of children"
    ],

    "Personal financial situation": [
        "Employment - 7 groups",
        "Can afford a meal with meat/chicken/fish every second day?",
        "Household structure",
        "Income quartiles",
        "Education completed"
    ],

    "How much trust the police?": [
        "Neighbourhood problems - crime, violence or vandalism",
        "How much trust the government?",
        "How much trust the legal system?",
        "Rural/urban living",
        "Age"
    ],

    "Deprivation index: No. of items hhold can't afford": [
        "Income quartiles",
        "Employment - 7 groups",
        "Can afford to keep home adequately warm?",
        "Household size",
        "Household structure"
    ],

    "Quality of education system?": [
        "Education completed",
        "How much trust the legal system?",
        "Income quartiles",
        "Rural/urban living",
        "Age"
    ],

    "The value of what I do is not recognised by others?": [
        "Employment - 7 groups",
        "How frequently participate in social activities?",
        "A person to get support from when feeling depressed",
        "Education completed",
        "Marital status"
    ],

    "Marital status": [
        "Age",
        "Household structure",
        "How frequently participate in social activities?",
        "No. of children",
        "Education completed"
    ],

    "Can most people be trusted?": [
        "Neighbourhood problems - crime, violence or vandalism",
        "How much trust the press?",
        "How much trust the government?",
        "Education completed",
        "Rural/urban living"
    ]
}

### Load data, recode variables, and define helper function


In [4]:
# Data
data = pd.read_csv('../data/data_eqls_LOOSE_filtered.csv')

# Reversing the scale on some features (higher values = more positive)
data["Health condition"] = 6 - data["Health condition"]
data["Can't find the way because life has become so complicated?"] = 6 - data["Can't find the way because life has become so complicated?"]
data["Household able to make ends meet?"] = 7 - data["Household able to make ends meet?"]
data["A person to get support from when feeling depressed"] = 3 - data["A person to get support from when feeling depressed"]
data["I feel I am free to decide how to live my life"] = 6 - data["I feel I am free to decide how to live my life"]
data["I generally feel that what I do in life is worthwhile"] = 6 - data["I generally feel that what I do in life is worthwhile"]
data["I am optimistic about the future"] = 6 - data["I am optimistic about the future"]

# Binary marital status: 1 = married/living with partner, 0 = all other categories
data["Marital status"] = (data["Marital status"] == 1).astype(int)

# Income subgroup variable
income_column = "Income quartiles"

# Important: Income quartiles is the subgroup variable here.
# Income quartiles is removed from the confounder sets within the income-specific models,
# because it is constant inside each subgroup.
def clean_confounders_for_income_subgroups(treatment, confounders):
    return [c for c in confounders if c not in [income_column, treatment]]


def calculate_causal_values(data_source, confounder_map):
    # Save results
    results = []

    # Iterate through all features
    for treatment, confounders in confounder_map.items():
        adjusted_confounders = clean_confounders_for_income_subgroups(treatment, confounders)

        try:
            print(f"Treatment: {treatment}")
            model = CausalModel(
                data=data_source,
                treatment=treatment,
                outcome=outcome,
                common_causes=adjusted_confounders
            )

            identified_estimand = model.identify_effect()

            estimate = model.estimate_effect(
                identified_estimand,
                method_name="backdoor.linear_regression"
            )

            results.append({
                "Treatment": treatment,
                "ATE (DML)": estimate.value,
                "Confounders used": adjusted_confounders
            })

            print("ATE:", estimate.value)
            print("Confounders used:", adjusted_confounders)
            print("-" * 80)

        except Exception as e:
            print(f"Error for treatment {treatment}: {e}")
            print("-" * 80)

            results.append({
                "Treatment": treatment,
                "ATE (DML)": None,
                "Confounders used": adjusted_confounders
            })

    return results


### Create income subgroups


In [5]:
# Income subgroups
df_q1 = data[data[income_column] == 1].copy()  # 1st quartile
df_q2 = data[data[income_column] == 2].copy()  # 2nd quartile
df_q3 = data[data[income_column] == 3].copy()  # 3rd quartile
df_q4 = data[data[income_column] == 4].copy()  # 4th quartile

income_subgroups = {
    "q1": df_q1,
    "q2": df_q2,
    "q3": df_q3,
    "q4": df_q4,
}

for group_name, group_df in income_subgroups.items():
    print(group_name, group_df.shape)


q1 (1170, 168)
q2 (1430, 168)
q3 (1681, 168)
q4 (1841, 168)


### Calculate ATEs for each income subgroup


In [6]:
income_results = {}

for group_name, group_df in income_subgroups.items():
    print(f"\n===== {group_name} =====")
    df_results = pd.DataFrame(calculate_causal_values(group_df, feature_confounder_map))
    df_results.sort_values(by="ATE (DML)", ascending=False, inplace=True, key=abs)
    income_results[group_name] = df_results
    display(df_results)



===== q1 =====
Treatment: Health condition
ATE: 0.6638867585268073
Confounders used: ['Age', 'Chronic health problems?', 'Education completed', 'Employment - 7 groups']
--------------------------------------------------------------------------------
Treatment: I generally feel that what I do in life is worthwhile
ATE: 0.5978923215674472
Confounders used: ['A person to get support from when feeling depressed', 'How frequently participate in social activities?', 'Employment - 7 groups', 'Marital status', 'Health condition']
--------------------------------------------------------------------------------
Treatment: Can't find the way because life has become so complicated?
ATE: -0.5480249657195362
Confounders used: ['Education completed', 'Employment - 7 groups', 'A person to get support from when feeling depressed', 'Age', 'Household size']
--------------------------------------------------------------------------------
Treatment: I feel I am free to decide how to live my life
ATE: 0.47

C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future vers

,Treatment,ATE (DML),Confounders used
6,Personal financial situation,0.707126,"[Employment - 7 groups, Can afford a meal with..."
0,Health condition,0.663887,"[Age, Chronic health problems?, Education comp..."
1,I generally feel that what I do in life is wor...,0.597892,[A person to get support from when feeling dep...
2,Can't find the way because life has become so ...,-0.548025,"[Education completed, Employment - 7 groups, A..."
11,Marital status,0.534504,"[Age, Household structure, How frequently part..."
3,I feel I am free to decide how to live my life,0.478847,"[Education completed, How much trust the gover..."
5,Household able to make ends meet?,0.464519,"[Employment - 7 groups, Personal financial sit..."
4,I am optimistic about the future,0.456428,"[Personal financial situation, Access to recre..."
10,The value of what I do is not recognised by ot...,0.454709,"[Employment - 7 groups, How frequently partici..."
8,Deprivation index: No. of items hhold can't af...,-0.407758,"[Employment - 7 groups, Can afford to keep hom..."



===== q2 =====
Treatment: Health condition


C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]


...

ATE: 0.08825324347241548
Confounders used: ['Neighbourhood problems - crime, violence or vandalism', 'How much trust the government?', 'How much trust the legal system?', 'Rural/urban living', 'Age']
--------------------------------------------------------------------------------
Treatment: Deprivation index: No. of items hhold can't afford


C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]


ATE: -0.3313472883474935
Confounders used: ['Employment - 7 groups', 'Can afford to keep home adequately warm?', 'Household size', 'Household structure']
--------------------------------------------------------------------------------
Treatment: Quality of education system?
ATE: 0.17784648814111748
Confounders used: ['Education completed', 'How much trust the legal system?', 'Rural/urban living', 'Age']
--------------------------------------------------------------------------------
Treatment: The value of what I do is not recognised by others?
ATE: 0.3148960843551052
Confounders used: ['Employment - 7 groups', 'How frequently participate in social activities?', 'A person to get support from when feeling depressed', 'Education completed', 'Marital status']
--------------------------------------------------------------------------------
Treatment: Marital status
ATE: 0.401769149551372
Confounders used: ['Age', 'Household structure', 'How frequently participate in social activities?', 'N

C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future vers

,Treatment,ATE (DML),Confounders used
0,Health condition,0.651404,"[Age, Chronic health problems?, Education comp..."
1,I generally feel that what I do in life is wor...,0.530783,[A person to get support from when feeling dep...
3,I feel I am free to decide how to live my life,0.433147,"[Education completed, How much trust the gover..."
11,Marital status,0.401769,"[Age, Household structure, How frequently part..."
2,Can't find the way because life has become so ...,-0.350821,"[Education completed, Employment - 7 groups, A..."
8,Deprivation index: No. of items hhold can't af...,-0.331347,"[Employment - 7 groups, Can afford to keep hom..."
6,Personal financial situation,0.327706,"[Employment - 7 groups, Can afford a meal with..."
10,The value of what I do is not recognised by ot...,0.314896,"[Employment - 7 groups, How frequently partici..."
4,I am optimistic about the future,0.302058,"[Personal financial situation, Access to recre..."
5,Household able to make ends meet?,0.267535,"[Employment - 7 groups, Personal financial sit..."


### Combined Results

In [7]:
# Create one comparison table with one ATE column per income subgroup
comparison_tables = []

for group_name, df_results in income_results.items():
    temp = df_results[["Treatment", "ATE (DML)"]].rename(
        columns={"ATE (DML)": f"ATE_{group_name}"}
    )
    comparison_tables.append(temp)

# Merge all income-specific result tables
from functools import reduce

df_compare_income = reduce(
    lambda left, right: pd.merge(left, right, on="Treatment", how="outer"),
    comparison_tables
)

# Sort by mean absolute ATE across income subgroups
ate_columns = [col for col in df_compare_income.columns if col.startswith("ATE_")]
df_compare_income["Mean_abs_ATE"] = df_compare_income[ate_columns].abs().mean(axis=1)
df_compare_income = df_compare_income.sort_values("Mean_abs_ATE", ascending=False)

display(df_compare_income)


,Treatment,ATE_q1,ATE_q2,ATE_q3,ATE_q4,Mean_abs_ATE
3,Health condition,0.663887,0.648783,0.637180,0.651404,0.650314
8,I generally feel that what I do in life is wor...,0.597892,0.567759,0.655613,0.530783,0.588012
10,Personal financial situation,0.707126,0.611921,0.379867,0.327706,0.506655
7,I feel I am free to decide how to live my life,0.478847,0.567560,0.511380,0.433147,0.497733
9,Marital status,0.534504,0.497565,0.467232,0.401769,0.475268
1,Can't find the way because life has become so ...,-0.548025,-0.432974,-0.381316,-0.350821,0.428284
6,I am optimistic about the future,0.456428,0.390866,0.399441,0.302058,0.387198
12,The value of what I do is not recognised by ot...,0.454709,0.373972,0.381030,0.314896,0.381152
4,Household able to make ends meet?,0.464519,0.327612,0.310806,0.267535,0.342618
2,Deprivation index: No. of items hhold can't af...,-0.407758,-0.291343,-0.281192,-0.331347,0.327910


### Export Results

In [8]:
# Export combined results
import os
os.makedirs("Results", exist_ok=True)
df_compare_income.to_csv("Results/results_income_causal_analysis.csv", index=False)

# Optional: export each subgroup table separately
#for group_name, df_results in income_results.items():
#    df_results.to_csv(f"Results/results_{group_name}_causal_analysis.csv", index=False)
